# Le modèle de Toulmin (1958)

[← Dung_AF_Semantics](Argumentation-03-Dung-AF-Semantics-Python.ipynb) | [↑ ArgumentAnalysis](README.md) | [Value_Based_AF →](Argumentation-03b-Value-Based-AF-Python.ipynb) | [ADF_Semantiques →](Argumentation-03d-ADF-Semantiques-Python.ipynb)

## Pourquoi ce notebook

Les notebooks [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb), [Argumentation-03b-Value-Based-AF-Python](Argumentation-03b-Value-Based-AF-Python.ipynb) et [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) traitent l'argumentation **formelle/abstraite** : des arguments anonymes reliés par une relation d'attaque. Mais dans la pratique — un débat politique, un mémoire d'expertise, une plaidoirie — un argument n'est pas un nœud anonyme : il a une **structure interne**. Le philosophe **Stephen Toulmin** (1958, *The Uses of Argument*) a proposé une anatomie de cette structure, devenu l'outil de référence en analyse informelle de l'argumentation (rhétorique, droit, éthique appliquée, communication scientifique).

Toulmin observe qu'un argument complet ne se résume pas à « voici des faits, donc voici une conclusion ». Entre les faits (*Data*) et la conclusion (*Claim*), il y a toujours une **inférence autorisée** — le *Warrant* — qui dit *pourquoi* ces faits autorisent cette conclusion. Et ce warrant repose lui-même sur un *Backing* (un fondement), tolère un *Qualifier* (une modalité), et affronte des *Rebuttals* (des exceptions). Omettre ces pièces, c'est produire un argument qui *paraît* solide mais dont la chaîne logique est invisible — donc impossible à contester précisément.

Ce notebook reconstruit le **schéma à 6 composants** de Toulmin en pur C# (BCL .NET), montre comment **auditer la complétude** d'un argument, et — c'est le pont computationnel — comment **traduire un débat d'arguments Toulmin en un cadre de Dung** : les rebuttals deviennent des attaques, et les sémantiques (grounded, preferred) décident quels claims survivent.

**Série Argumentation** | | Précédent : [00-Setup-Tweety-Python](./Argumentation-00-Setup-Tweety-Python.ipynb) | Suivant : [01b-Schemes-Walton-Python](./Argumentation-01b-Schemes-Walton-Python.ipynb) | [↑ Série Argumentation](README.md) | [jumeau Python](Argumentation-01-Toulmin-Model-Python.ipynb)

## Le schéma à 6 composants

Un argument, selon Toulmin, se déploie en six pièces. Les trois premières sont **essentielles** (sans elles, il n'y a pas d'argument) ; les trois dernières le **qualifient** et le **durcissent**.

| Composant | Rôle | Question qu'il répond |
|-----------|------|------------------------|
| **Claim** (conclusion, *C*) | La thèse défendue | « Que veux-tu établir ? » |
| **Data** (données, *D*) | Les faits/observations invoqués | « Sur quoi t'appuies-tu ? » |
| **Warrant** (garant, *W*) | La règle d'inférence *D → C* | « Comment passes-tu de D à C ? » |
| **Backing** (fondement, *B*) | Le support du warrant (loi, autorité, théorie) | « Pourquoi cette règle vaut-elle ? » |
| **Qualifier** (modalité, *Q*) | La force de la conclusion (« nécessairement », « probablement ») | « Avec quelle certitude ? » |
| **Rebuttal** (réfutation, *R*) | Les conditions d'exception | « Quand ton argument tombe-t-il ? » |

La pièce **sous-enseignée** est le *Warrant*. Beaucoup d'arguments implicites sautent de *Data* à *Claim* sans expliciter la règle — et c'est précisément là que se logent les sophismes et les désaccords. Rendre le warrant explicite est l'acte analytique central du modèle Toulmin.

## Exemple concret : un argument complet

Pour rendre les composants concrets, déployons un argument familier :

> « *Le projet X doit probablement être financé, car son ROI projeté est de 15 % ; or les projets à ROI élevé méritent financement — principe d'allocation rationnelle du capital — sauf si la situation de défaut attendue le disqualifie. * »

Décomposé selon Toulmin :

- **Claim** : « Le projet X doit être financé »
- **Data** : « X a un ROI projeté de 15 % »
- **Warrant** : « Les projets à ROI élevé méritent financement »
- **Backing** : « Principe d'allocation rationnelle du capital »
- **Qualifier** : « probablement »
- **Rebuttal** (condition d'exception) : « sauf si le risque de défaut est prohibitif »

Implémentons ce schéma comme une classe C#, puis vérifions sa complétude.

In [1]:
#nullable enable
// --- Le schema de Toulmin (1958), BCL .NET uniquement ---
ToulminArgument.PrintExample();

public sealed class ToulminArgument
{
    public string Name { get; }
    public string Claim { get; }
    public string Data { get; }
    public string Warrant { get; }
    public string Backing { get; }
    public string Qualifier { get; }

    // rebuttal_targets = ensemble des NOMS d'arguments que celui-ci refute
    public HashSet<string> RebuttalTargets { get; }

    public ToulminArgument(
        string name,
        string claim,
        string data,
        string warrant,
        string backing = "",
        string qualifier = "",
        IEnumerable<string>? rebuttalTargets = null)
    {
        // Comme en Python, les valeurs sont conservees telles quelles :
        // aucune normalisation ne modifie les calculs de completude ou de ciblage.
        Name = name;
        Claim = claim;
        Data = data;
        Warrant = warrant;
        Backing = backing;
        Qualifier = qualifier;
        RebuttalTargets = rebuttalTargets is null
            ? new HashSet<string>(StringComparer.Ordinal)
            : new HashSet<string>(rebuttalTargets, StringComparer.Ordinal);
    }

    public bool IsComplete()
    {
        return !string.IsNullOrEmpty(Claim)
            && !string.IsNullOrEmpty(Data)
            && !string.IsNullOrEmpty(Warrant);
    }

    public List<string> MissingPieces()
    {
        var missing = new List<string>();
        if (string.IsNullOrEmpty(Claim))
        {
            missing.Add("claim");
        }

        if (string.IsNullOrEmpty(Data))
        {
            missing.Add("data");
        }

        if (string.IsNullOrEmpty(Warrant))
        {
            missing.Add("warrant");
        }

        return missing;
    }

    public string CompactRepresentation()
    {
        // Le .strip() de Python s'applique uniquement a la representation
        // compacte finale, pas aux champs utilises par les calculs.
        var prefix = string.IsNullOrEmpty(Qualifier) ? "" : Qualifier + " ";
        var claim = string.IsNullOrEmpty(Claim) ? "(incomplet)" : Claim;
        return $"[{Name}] {prefix}{claim}".Trim();
    }

    public string Describe()
    {
        var lines = new List<string>
        {
            $"[{Name}] Argument Toulmin (complet: {IsComplete()})",
            $"  Claim    : {Display(Claim, "(manquant)")}",
            $"  Data     : {Display(Data, "(manquant)")}",
            $"  Warrant  : {Display(Warrant, "(manquant)")}",
            $"  Backing  : {Display(Backing, "(non precise)")}",
            $"  Qualifier: {Display(Qualifier, "(non precise)")}"
        };

        var rebuttalTargets = RebuttalTargets.Count == 0
            ? "['(aucun)']"
            : $"[{string.Join(", ", RebuttalTargets.OrderBy(target => target, StringComparer.Ordinal).Select(target => $"'{target}'"))}]";
        lines.Add($"  Rebuttals cibles: {rebuttalTargets}");
        return string.Join(Environment.NewLine, lines);
    }

    private static string Display(string value, string fallback)
    {
        return string.IsNullOrEmpty(value) ? fallback : value;
    }

    public static void Require(bool condition, string message)
    {
        if (!condition)
        {
            throw new InvalidOperationException(message);
        }
    }

    public static void PrintExample()
    {
        // Deploiement de l'exemple "projet X".
        var argX = new ToulminArgument(
            name: "X",
            claim: "Le projet X doit etre finance",
            data: "X a un ROI projete de 15%",
            warrant: "Les projets a ROI eleve meritent financement",
            backing: "Principe d'allocation rationnelle du capital",
            qualifier: "probablement",
            // on ajoutera la cible plus tard, avec l'argument adverse
            rebuttalTargets: Array.Empty<string>());

        Console.WriteLine(argX.Describe());
        Console.WriteLine();
        var calculatedComplete = argX.IsComplete();
        var calculatedMissing = argX.MissingPieces();
        var expectedMissing = Array.Empty<string>();
        Require(calculatedComplete, "L'argument X doit etre complet.");
        Require(calculatedMissing.SequenceEqual(expectedMissing), "L'argument X ne doit avoir aucune piece manquante.");
        Console.WriteLine($"Verification X : complet calcule={calculatedComplete}, attendu=True ; pieces calculees={FormatList(calculatedMissing)}, attendues={FormatList(expectedMissing)} -> {(calculatedComplete && calculatedMissing.SequenceEqual(expectedMissing) ? "CONFORME" : "ECART")}");
        var calculatedCompact = argX.CompactRepresentation();
        var expectedCompact = "[X] probablement Le projet X doit etre finance";
        Require(calculatedCompact == expectedCompact, "La representation compacte de X est incorrecte.");
        Console.WriteLine($"Verification repr compacte : calcule='{calculatedCompact}', attendu='{expectedCompact}' -> {(calculatedCompact == expectedCompact ? "CONFORME" : "ECART")}");

        var argLeak = new ToulminArgument(
            name: "leak",
            claim: "X doit etre finance",
            data: "X a un ROI de 15%",
            warrant: "");
        var calculatedLeakCompact = argLeak.CompactRepresentation();
        var expectedLeakCompact = "[leak] X doit etre finance";
        Require(calculatedLeakCompact == expectedLeakCompact, "La representation compacte de leak doit conserver le claim.");
        Console.WriteLine($"Verification repr compacte leak : calcule='{calculatedLeakCompact}', attendu='{expectedLeakCompact}' -> {(calculatedLeakCompact == expectedLeakCompact ? "CONFORME" : "ECART")}");
    }

    internal static string FormatList(IEnumerable<string> values)
    {
        return $"[{string.Join(", ", values.Select(value => $"'{value}'"))}]";
    }
}

[X] Argument Toulmin (complet: True)
  Claim    : Le projet X doit etre finance
  Data     : X a un ROI projete de 15%
  Warrant  : Les projets a ROI eleve meritent financement
  Backing  : Principe d'allocation rationnelle du capital
  Qualifier: probablement
  Rebuttals cibles: ['(aucun)']


Verification X : complet calcule=True, attendu=True ; pieces calculees=[], attendues=[] -> CONFORME


Verification repr compacte : calcule='[X] probablement Le projet X doit etre finance', attendu='[X] probablement Le projet X doit etre finance' -> CONFORME


Verification repr compacte leak : calcule='[leak] X doit etre finance', attendu='[leak] X doit etre finance' -> CONFORME


## Auditer la complétude

Le piège classique de l'argumentation implicite : un énoncé qui *ressemble* à un argument mais saute le warrant. « *X a un ROI de 15 %, donc X doit être financé* » omet la règle d'inférence — le lecteur doit la deviner. La méthode `MissingPieces()` détecte ces trous, ce qui est précisément le travail d'un auditeur d'argumentation.

In [2]:
// Un argument INCOMPLET (warrant implicite) : diagnostic automatique.
var argLeak = new ToulminArgument(
    name: "leak",
    claim: "X doit etre finance",
    data: "X a un ROI de 15%",
    warrant: ""  // le warrant est implicite -> trou analytique
);

Console.WriteLine(argLeak.Describe());
Console.WriteLine();
var missing = argLeak.MissingPieces();
ToulminArgument.Require(!argLeak.IsComplete(), "L'argument leak doit etre incomplet.");
ToulminArgument.Require(missing.SequenceEqual(new[] { "warrant" }), "La piece manquante de leak doit etre exactement 'warrant'.");
if (missing.Count > 0)
{
    Console.WriteLine($">>> Argument INCOMPLET : piece(s) essentielle(s) manquante(s) = {ToulminArgument.FormatList(missing)}");
    Console.WriteLine(">>> Le warrant (D -> C) doit etre explicite : sur quelle regle d'inference passe-t-on du ROI a l'obligation de financer ?");
}
else
{
    Console.WriteLine(">>> Argument complet.");
}

[leak] Argument Toulmin (complet: False)
  Claim    : X doit etre finance
  Data     : X a un ROI de 15%
  Warrant  : (manquant)
  Backing  : (non precise)
  Qualifier: (non precise)
  Rebuttals cibles: ['(aucun)']


>>> Argument INCOMPLET : piece(s) essentielle(s) manquante(s) = ['warrant']


>>> Le warrant (D -> C) doit etre explicite : sur quelle regle d'inference passe-t-on du ROI a l'obligation de financer ?


## Le pont vers Dung : rebuttals = attaques

Voici le saut computationnel. Le modèle Toulmin décrit **un** argument isolé. Mais un débat réel en **confronte plusieurs** — et ces arguments se *réfutent* mutuellement (le composant Rebuttal). Or « *A réfute B* » est exactement « *A attaque B* » au sens de [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb). Donc :

1. On déploie chaque argument en sa forme Toulmin (6 composants).
2. On lit les `RebuttalTargets` de chacun → on construit la relation d'attaque du cadre de Dung.
3. On applique la sémantique *grounded* : les claims qui survivent sont ceux défendus dans le débat.

Construisons un débat à deux arguments qui se réfutent mutuellement, puis regardons le verdict du grounded.

In [3]:
// Primitives de Dung (comme Dung_AF_Semantics / Value_Based_AF).
DungDemo.Run();

public sealed class AF
{
    public HashSet<string> Args { get; }
    public HashSet<(string Attacker, string Target)> Attacks { get; }

    public AF(IEnumerable<string> args, IEnumerable<(string Attacker, string Target)> attacks)
    {
        Args = new HashSet<string>(args, StringComparer.Ordinal);
        Attacks = new HashSet<(string, string)>(attacks);
    }

    public IEnumerable<string> Attackers(string target)
    {
        return Attacks
            .Where(attack => attack.Target == target)
            .Select(attack => attack.Attacker)
            .Distinct(StringComparer.Ordinal)
            .ToHashSet(StringComparer.Ordinal);
    }
}

public static class DungSemantics
{
    public static bool Defeats(AF af, ISet<string> extension, string target)
    {
        return af.Attacks.Any(attack =>
            extension.Contains(attack.Attacker)
            && attack.Target == target);
    }

    public static bool Defends(AF af, ISet<string> extension, string argument)
    {
        return af.Attackers(argument).All(attacker => Defeats(af, extension, attacker));
    }

    public static HashSet<string> Grounded(AF af)
    {
        var extension = new HashSet<string>(StringComparer.Ordinal);
        while (true)
        {
            var candidates = af.Args
                .Where(argument => Defends(af, extension, argument))
                .ToHashSet(StringComparer.Ordinal);
            var next = new HashSet<string>(extension, StringComparer.Ordinal);
            next.UnionWith(candidates);

            if (next.SetEquals(extension))
            {
                return extension;
            }

            extension = next;
        }
    }
}

public static class ToulminDebate
{
    public static AF ToAf(IEnumerable<ToulminArgument> arguments)
    {
        var argumentList = arguments.ToList();
        var names = argumentList
            .Select(argument => argument.Name)
            .ToHashSet(StringComparer.Ordinal);
        var attacks = new HashSet<(string Attacker, string Target)>();

        foreach (var argument in argumentList)
        {
            foreach (var target in argument.RebuttalTargets)
            {
                if (names.Contains(target))
                {
                    attacks.Add((argument.Name, target));
                }
            }
        }

        return new AF(names, attacks);
    }
}

public static class DungDemo
{
    public static void Run()
    {
        var argumentA = new ToulminArgument(
            name: "A",
            claim: "Le projet X doit etre finance",
            data: "X a un ROI projete de 15%",
            warrant: "Les projets a ROI eleve meritent financement",
            backing: "Principe d'allocation rationnelle du capital",
            qualifier: "probablement",
            rebuttalTargets: new[] { "B" });

        var argumentB = new ToulminArgument(
            name: "B",
            claim: "Le projet X ne doit pas etre finance",
            data: "X presente un risque de defaut de 40%",
            warrant: "Les projets trop risques doivent etre evites",
            backing: "Principe de prudence",
            qualifier: "possiblement",
            rebuttalTargets: new[] { "A" });

        Console.WriteLine("=== Debat a 2 arguments (rebuttal mutuel) ===");
        foreach (var argument in new[] { argumentA, argumentB })
        {
            Console.WriteLine(argument.Describe());
            Console.WriteLine();
        }

        var af2 = ToulminDebate.ToAf(new[] { argumentA, argumentB });
        var expectedArgs2 = new HashSet<string>(new[] { "A", "B" }, StringComparer.Ordinal);
        var expectedAttacks2 = new HashSet<(string Attacker, string Target)>
        {
            ("A", "B"),
            ("B", "A")
        };
        ToulminArgument.Require(af2.Args.SetEquals(expectedArgs2), "Les noeuds du cadre Dung a deux arguments sont incorrects.");
        ToulminArgument.Require(af2.Attacks.SetEquals(expectedAttacks2), "Les attaques du cadre Dung a deux arguments sont incorrectes.");
        Console.WriteLine($"Cadre de Dung : args={FormatArguments(af2)}, attacks={FormatAttacks(af2)}");
        var grounded2 = DungSemantics.Grounded(af2);
        ToulminArgument.Require(grounded2.Count == 0, "Le cadre a deux arguments doit produire un grounded vide.");
        Console.WriteLine(grounded2.Count == 0
            ? "Extension grounded : {} (vide)"
            : $"Extension grounded : {FormatArguments(grounded2)}");
        Console.WriteLine(">>> Verdict : A<->B forment un cycle. Ni l'un ni l'autre ne defend l'autre");
        Console.WriteLine("    -> grounded VIDE = standoff, aucune conclusion ne survit seule.");
    }

    private static string FormatArguments(AF af)
    {
        return FormatNames(af.Args);
    }

    private static string FormatArguments(IEnumerable<string> arguments)
    {
        return FormatNames(arguments);
    }

    private static string FormatNames(IEnumerable<string> arguments)
    {
        return $"[{string.Join(", ", arguments.OrderBy(name => name, StringComparer.Ordinal).Select(name => $"'{name}'"))}]";
    }

    private static string FormatAttacks(AF af)
    {
        var attacks = af.Attacks
            .OrderBy(attack => attack.Attacker, StringComparer.Ordinal)
            .ThenBy(attack => attack.Target, StringComparer.Ordinal)
            .Select(attack => $"('{attack.Attacker}', '{attack.Target}')");
        return $"[{string.Join(", ", attacks)}]";
    }
}

=== Debat a 2 arguments (rebuttal mutuel) ===


[A] Argument Toulmin (complet: True)
  Claim    : Le projet X doit etre finance
  Data     : X a un ROI projete de 15%
  Warrant  : Les projets a ROI eleve meritent financement
  Backing  : Principe d'allocation rationnelle du capital
  Qualifier: probablement
  Rebuttals cibles: ['B']


[B] Argument Toulmin (complet: True)
  Claim    : Le projet X ne doit pas etre finance
  Data     : X presente un risque de defaut de 40%
  Warrant  : Les projets trop risques doivent etre evites
  Backing  : Principe de prudence
  Qualifier: possiblement
  Rebuttals cibles: ['A']


Cadre de Dung : args=['A', 'B'], attacks=[('A', 'B'), ('B', 'A')]


Extension grounded : {} (vide)


>>> Verdict : A<->B forment un cycle. Ni l'un ni l'autre ne defend l'autre


    -> grounded VIDE = standoff, aucune conclusion ne survit seule.


## Le verdict bascule : introduire un arbitre

Le standoff (grounded vide) est insatisfaisant : le débat est *indécis*. Mais ajoutons un **troisième argument** *C* qui réfute *B* seulement (sans toucher à *A*). Concrètement, *C* conteste le *Data* de *B* (le risque de défaut est sur-estimé). Le cycle se brise : *C* défait *B*, donc *B* n'attaque plus *A*, donc *A* redevient défendable.

In [4]:
// On ajoute un 3e argument C qui refute B seulement (arbitre).
// A et B sont re-instancies ici pour rejouer le meme debat avec l'arbitre C en plus.
var argumentA = new ToulminArgument(
    name: "A",
    claim: "Le projet X doit etre finance",
    data: "X a un ROI projete de 15%",
    warrant: "Les projets a ROI eleve meritent financement",
    backing: "Principe d'allocation rationnelle du capital",
    qualifier: "probablement",
    rebuttalTargets: new[] { "B" });

var argumentB = new ToulminArgument(
    name: "B",
    claim: "Le projet X ne doit pas etre finance",
    data: "X presente un risque de defaut de 40%",
    warrant: "Les projets trop risques doivent etre evites",
    backing: "Principe de prudence",
    qualifier: "possiblement",
    rebuttalTargets: new[] { "A" });

var argumentC = new ToulminArgument(
    name: "C",
    claim: "Le risque de defaut de X est surestime",
    data: "L'historique montre 10% de defaut reel sur projets similaires",
    warrant: "Les projections de risque doivent etre calibrees sur donnees passees",
    backing: "Principe de calibration empirique",
    qualifier: "fortement",
    rebuttalTargets: new[] { "B" });

Console.WriteLine("=== Debat a 3 arguments (C arbitre en refutant B) ===");
Console.WriteLine(argumentC.Describe());
Console.WriteLine();

var af3 = ToulminDebate.ToAf(new[] { argumentA, argumentB, argumentC });
var expectedArgs3 = new HashSet<string>(new[] { "A", "B", "C" }, StringComparer.Ordinal);
var expectedAttacks3 = new HashSet<(string Attacker, string Target)>
{
    ("A", "B"),
    ("B", "A"),
    ("C", "B")
};
ToulminArgument.Require(af3.Args.SetEquals(expectedArgs3), "Les noeuds du cadre Dung a trois arguments sont incorrects.");
ToulminArgument.Require(af3.Attacks.SetEquals(expectedAttacks3), "Les attaques du cadre Dung a trois arguments sont incorrectes.");
Console.WriteLine($"Cadre de Dung : args={FormatNames(af3.Args)}");
Console.WriteLine($"                attacks={FormatAttacks(af3.Attacks)}");
var grounded3 = DungSemantics.Grounded(af3);
var expectedGrounded3 = new HashSet<string>(new[] { "A", "C" }, StringComparer.Ordinal);
ToulminArgument.Require(grounded3.SetEquals(expectedGrounded3), "Le cadre a trois arguments doit produire le grounded {A, C}.");
Console.WriteLine($"Extension grounded : {FormatNames(grounded3)}");
Console.WriteLine();
Console.WriteLine(">>> Lecture : C defait B -> B ne peut plus attaquer A -> A est defendu.");
Console.WriteLine("    Les claims A et C survivent ; B (le contre-argument) est exclu.");
Console.WriteLine("    Le warrant sous-jacent : un seul arbitre peut faire basculer un debat");
Console.WriteLine("    indécis vers une conclusion, SANS toucher aux arguments principaux.");

static string FormatNames(IEnumerable<string> names)
{
    var ordered = names.OrderBy(name => name, StringComparer.Ordinal).Select(name => $"'{name}'");
    return $"[{string.Join(", ", ordered)}]";
}

static string FormatAttacks(IEnumerable<(string Attacker, string Target)> attacks)
{
    var ordered = attacks
        .OrderBy(attack => attack.Attacker, StringComparer.Ordinal)
        .ThenBy(attack => attack.Target, StringComparer.Ordinal)
        .Select(attack => $"('{attack.Attacker}', '{attack.Target}')");
    return $"[{string.Join(", ", ordered)}]";
}

=== Debat a 3 arguments (C arbitre en refutant B) ===


[C] Argument Toulmin (complet: True)
  Claim    : Le risque de defaut de X est surestime
  Data     : L'historique montre 10% de defaut reel sur projets similaires
  Warrant  : Les projections de risque doivent etre calibrees sur donnees passees
  Backing  : Principe de calibration empirique
  Qualifier: fortement
  Rebuttals cibles: ['B']


Cadre de Dung : args=['A', 'B', 'C']


                attacks=[('A', 'B'), ('B', 'A'), ('C', 'B')]


Extension grounded : ['A', 'C']


>>> Lecture : C defait B -> B ne peut plus attaquer A -> A est defendu.


    Les claims A et C survivent ; B (le contre-argument) est exclu.


    Le warrant sous-jacent : un seul arbitre peut faire basculer un debat


    indécis vers une conclusion, SANS toucher aux arguments principaux.


## Vérifiez votre compréhension

Les trois exercices approfondissent le modèle. Chaque stub est à compléter — le notebook s'exécute de bout en bout même non complété (les exercices renvoient `null`/valeurs neutres et affichent un message). La méthode de chaque exercice est déjà démontrée sur un exemple dans le carnet :

- **Exercice 1** (diagnostiquer un argument incomplet) → la section *Auditer la complétude* l'a fait sur l'argument « leak » : la complétion calculée est `False` et la liste calculée contient exactement `['warrant']`.
- **Exercice 2** (construire un standoff) → la section *Le pont vers Dung* l'a fait avec A et B : l'extension grounded calculée est vide, `{}`.
- **Exercice 3** (trouver l'arbitre qui bascule le débat) → la section *Le verdict bascule* l'a fait avec C : l'extension grounded calculée contient exactement `['A', 'C']`.

Les blocs de vérification des trois exercices sont exécutables : tant que le stub renvoie une réponse neutre, le carnet affiche « Exercice n a completer » ; lorsqu'une réponse est fournie, les contrôles vérifient sa complétude et sa cohérence avec le résultat attendu. Pour l'exercice 3, le contrôle utilise le nom réel de la cible, exige que l'arbitre soit un nouveau nœud qui n'attaque que cette cible, puis recalcule le grounded du cadre à trois arguments et compare cet ensemble à celui renvoyé.

### Exercice 1 — Diagnostiquer un argument incomplet

Étant donné un énoncé familier d'opinion, identifiez la pièce essentielle manquante et complétez l'argument Toulmin pour qu'il soit `IsComplete() == true`.

**Objectif** : prendre l'énoncé « *Il pleut, donc je prends mon parapluie* », le modéliser (claim + data fournis, warrant à expliciter), et le rendre complet.

In [5]:
#nullable enable
// Exercice 1 : a completer
public static class Exercices1
{
    /// <summary>
    /// Renvoie un ToulminArgument complet modelisant : 'Il pleut, donc je
    /// prends mon parapluie'.
    ///
    /// Indice : le DATA = 'il pleut', le CLAIM = 'je prends mon parapluie'.
    /// Etape 1 : quel est le WARRANT (regle D -> C) implicite ? L'expliciter.
    /// Etape 2 : construire le ToulminArgument avec les 3 composants essentiels.
    /// </summary>
    public static ToulminArgument? ArgumentParapluieComplet()
    {
        // TODO etudiant
        return null;
    }
}

var arg = Exercices1.ArgumentParapluieComplet();
if (arg is null)
{
    Console.WriteLine("Exercice 1 a completer.");
}
else
{
    ToulminArgument.Require(arg.IsComplete(), "La reponse doit etre un argument complet.");
    var miss = arg.MissingPieces();
    ToulminArgument.Require(miss.Count == 0, "La reponse ne doit avoir aucune piece manquante.");
    Console.WriteLine(arg.Describe());
    Console.WriteLine("SUCCES : argument complet (warrant explicite).");
}

Exercice 1 a completer.


### Exercice 2 — Construire un débat équilibré (standoff)

Construisez **deux** arguments Toulmin complets qui se réfutent mutuellement (comme A et B ci-dessus), sur un sujet de votre choix (ex. « *le télétravail devrait être la norme* »). Vérifiez via `ToulminDebate.ToAf` + `DungSemantics.Grounded` que le débat est un **standoff** (grounded vide).

**Objectif** : sentir que deux arguments *corrects* mais opposés produisent une indécision — c'est pourquoi un débat pur pro/contra est rarement tranché sans un troisième élément.

In [6]:
#nullable enable
// Exercice 2 : a completer
public static class Exercices2
{
    /// <summary>
    /// Construisez deux ToulminArgument complets (PRO et CONTRA le teletravail
    /// comme norme) qui se refutent mutuellement. Renvoie (pro, contra).
    ///
    /// Indice : chaque argument doit avoir claim + data + warrant non vides, et
    /// RebuttalTargets pointant vers le nom de l'autre.
    /// </summary>
    public static (ToulminArgument? Pro, ToulminArgument? Contra) DebatTeletravailStandoff()
    {
        // TODO etudiant
        return (null, null);
    }
}

var (pro, contra) = Exercices2.DebatTeletravailStandoff();
if (pro is null || contra is null)
{
    Console.WriteLine("Exercice 2 a completer.");
}
else
{
    ToulminArgument.Require(pro.IsComplete(), "L'argument PRO doit etre complet.");
    ToulminArgument.Require(contra.IsComplete(), "L'argument CONTRA doit etre complet.");
    ToulminArgument.Require(pro.Name != contra.Name, "Les deux arguments doivent avoir des noms distincts.");
    ToulminArgument.Require(pro.RebuttalTargets.Contains(contra.Name), "PRO doit refuter CONTRA.");
    ToulminArgument.Require(contra.RebuttalTargets.Contains(pro.Name), "CONTRA doit refuter PRO.");

    var afEx2 = ToulminDebate.ToAf(new[] { pro, contra });
    var gEx2 = DungSemantics.Grounded(afEx2);
    ToulminArgument.Require(gEx2.Count == 0, "Le debat doit produire un grounded vide (standoff).");
    var attaques = afEx2.Attacks
        .OrderBy(attack => attack.Attacker, StringComparer.Ordinal)
        .ThenBy(attack => attack.Target, StringComparer.Ordinal)
        .Select(attack => $"('{attack.Attacker}', '{attack.Target}')");
    Console.WriteLine($"Attaques : [{string.Join(", ", attaques)}]");
    Console.WriteLine("Grounded : {} (vide = standoff)");
    Console.WriteLine("SUCCES : standoff confirme (debat equilibre).");
}

Exercice 2 a completer.


### Exercice 3 — Trouver l'arbitre qui fait basculer le débat

Reprenant votre débat de l'exercice 2 (pro/contra en standoff), construisez un **troisième** argument *C* qui réfute *un seul* des deux, de sorte que le grounded passe de vide à une conclusion non vide. Identifiez lequel des deux claims survit.

**Objectif** : comprendre que la conclusion d'un débat Toulmin-Dung dépend de *où* l'arbitre frappe — réfuter *pro* fait gagner *contra*, et inversement.

In [7]:
#nullable enable
// Exercice 3 : a completer
public static class Exercices3
{
    /// <summary>
    /// Construit un 3e argument C qui refute exactement l'argument dont le nom
    /// est `cible`, puis renvoie (C, ensemble_grounded_obtenu).
    ///
    /// `cible` doit etre le nom reel de `pro` ou de `contra`. Le nom de C doit
    /// differer des deux autres, et RebuttalTargets doit contenir uniquement
    /// le nom de l'argument cible.
    /// </summary>
    public static (ToulminArgument? Arbitre, HashSet<string>? Grounded) ArbitreQuiBascule(
        ToulminArgument pro,
        ToulminArgument contra,
        string cible)
    {
        // TODO etudiant
        return (null, null);
    }
}

var (proEx3, contraEx3) = Exercices2.DebatTeletravailStandoff();
if (proEx3 is null || contraEx3 is null)
{
    Console.WriteLine("Exercice 3 a completer (depend de l'exercice 2).");
}
else
{
    var (arbitre, grounded) = Exercices3.ArbitreQuiBascule(proEx3, contraEx3, contraEx3.Name);
    if (arbitre is null || grounded is null)
    {
        Console.WriteLine("Exercice 3 a completer.");
    }
    else
    {
        ToulminArgument.Require(arbitre.IsComplete(), "L'arbitre doit etre un argument complet.");
        ToulminArgument.Require(arbitre.Name != proEx3.Name && arbitre.Name != contraEx3.Name, "L'arbitre doit etre un nouveau noeud du debat.");
        ToulminArgument.Require(arbitre.RebuttalTargets.Count == 1, "L'arbitre doit attaquer exactement un seul argument.");
        ToulminArgument.Require(arbitre.RebuttalTargets.SetEquals(new[] { contraEx3.Name }), "L'arbitre doit attaquer uniquement la cible choisie.");

        var cadreEx3 = ToulminDebate.ToAf(new[] { proEx3, contraEx3, arbitre });
        var groundedCalcule = DungSemantics.Grounded(cadreEx3);
        ToulminArgument.Require(grounded.SetEquals(groundedCalcule), "L'ensemble grounded renvoye doit etre exactement celui calcule sur le cadre construit.");
        ToulminArgument.Require(groundedCalcule.Count > 0, "Le debat avec arbitre doit produire un grounded non vide.");
        ToulminArgument.Require(groundedCalcule.Contains(arbitre.Name), "L'arbitre doit appartenir au grounded.");
        ToulminArgument.Require(groundedCalcule.Contains(proEx3.Name), "Le claim non refute doit appartenir au grounded.");
        ToulminArgument.Require(!groundedCalcule.Contains(contraEx3.Name), "Le claim refute ne doit pas appartenir au grounded.");
        var noms = $"[{string.Join(", ", groundedCalcule.OrderBy(name => name, StringComparer.Ordinal).Select(name => $"'{name}'"))}]";
        Console.WriteLine($"Arbitre {arbitre.Name} refute {contraEx3.Name} -> grounded = {noms}");
        Console.WriteLine("SUCCES : le debat bascule et le claim refute est exclu.");
    }
}

Exercice 3 a completer (depend de l'exercice 2).


## À retenir

- Un argument n'est pas un bloc monolithique : le **schéma de Toulmin** le déploie en six pièces, dont trois **essentielles** (claim, data, warrant). Le *warrant* — la règle d'inférence *D → C* — est la pièce le plus souvent laissée implicite : l'audit l'a pris sur le vif avec l'argument « leak », dont la complétion calculée est `False` et la liste des pièces manquantes contient exactement `['warrant']`, là où l'argument complet `[X]` a une complétion calculée `True` et aucune pièce manquante.
- Auditer la complétude (`IsComplete` / `MissingPieces`) est un acte analytique : il force à expliciter la chaîne logique plutôt que la présupposer — et le diagnostic imprimé dit *pourquoi* le warrant manque : « sur quelle règle d'inférence passe-t-on du ROI à l'obligation de financer ? »
- **Le pont vers Dung** : en déployant plusieurs arguments Toulmin et en lisant leurs rebuttals comme des attaques, on obtient un cadre de Dung (`args=['A', 'B']`, `attacks=[('A', 'B'), ('B', 'A')]`). Les sémantiques tranchent alors le débat : un standoff (`Extension grounded : {} (vide)`) signifie que le débat est *structurellement* indécis.
- Un seul arbitre suffit à faire basculer un débat indécis : en ajoutant C qui attaque B, l'extension grounded passe de `{}` à `['A', 'C']` — sans toucher aux arguments principaux. C'est la prémisse du warrant « un seul arbitre peut faire basculer un débat indécis » que le carnet a mesurée plutôt qu'affirmée.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place du modèle de Toulmin dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Argumentation dialectique (ADF)** : [Argumentation-03d-ADF-Semantiques-Python](Argumentation-03d-ADF-Semantiques-Python.ipynb) remplace l'attaque binaire par une *condition d'acceptation* booléenne sur les parents ; le résultat d'une sémantique y devient une interprétation à valeurs `t` (accepté) / `f` (refusé) / `u` (indécis) — là où le `grounded` de Dung laisse un cadre entièrement vide, l'ADF laisse des arguments *indécis*.

**Argumentation graduée** : plutôt qu'un verdict accepté/rejeté, [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) attribue une *force* numérique à chaque argument — utile quand un débat ne se résout pas par grounded seul.

**Valeurs et audiences** : les warrants reposent souvent sur des *valeurs* (cf. [Argumentation-03b-Value-Based-AF-Python](Argumentation-03b-Value-Based-AF-Python.ipynb)) ; deux audiences aux valeurs différentes peuvent accepter le même *Data* mais rejeter le même *Warrant* — d'où l'intérêt de croiser Toulmin (structure) et VAF (préférences).

**Argumentation bipolaire** : Toulmin distingue l'attaque (rebuttal) du *support* implicite entre data et claim ; les cadres bipolaires (Cayrol & Lagasquie-Schiex, 2005) modélisent explicitement une relation de support en plus de l'attaque.

**Référence** — Stephen Toulmin, *The Uses of Argument* (Cambridge University Press, 1958, mis à jour 2003), le livre fondateur du schéma à six composants reconstruit ici.

***

### Attestation de parité Python / C#

La comparaison des résultats du carnet Python source et du jumeau C# confirme les mêmes instances et les mêmes valeurs de référence :

| Cas | Python — carnet source | C# — carnet jumeau |
|---|---|---|
| Argument `X` | complet ; pièces manquantes vides ; représentation `[X] probablement Le projet X doit etre finance` | mêmes complétude, liste vide et représentation compacte |
| Argument `leak` | incomplet ; pièce manquante `warrant` ; représentation `[leak] X doit etre finance` | mêmes incomplétude, pièce manquante et représentation compacte |
| Débat A/B | arguments `{A, B}` ; attaques `{A→B, B→A}` ; grounded vide | mêmes ensembles de nœuds, d'attaques et extension grounded vide |
| Débat A/B/C | arguments `{A, B, C}` ; attaques `{A→B, B→A, C→B}` ; grounded `{A, C}` | mêmes ensembles de nœuds, d'attaques et extension grounded `{A, C}` |

Les trois exercices restent des stubs neutres : aucune solution d'étudiant n'est révélée, et leurs contrôles ne valident une réponse que lorsqu'elle est effectivement fournie.

*Notebook pédagogique — BCL .NET uniquement, déterministe, sans LLM ni solveur externe. Conforme C.1 (stubs sans erreur volontaire), C.2 (sorties réelles), exercices à compléter.*